# ITSN1 Gene Variants and their Association with Parkinson's Disease
- Project: ITSN1 Burden
- Date Created: 7/10/25
- Date Last Modified: 3/11/26
- Data: All of Us Release 8


This project explores the *ITSN1* gene variants in AoU Data.

## Imports

In [ ]:
# System
import os
import sys

# Dataframe and array manipulation
import pandas as pd
import openpyxl
import numpy as np
import math

# Path management
import pathlib
from pathlib import Path
import glob

# Statistics
import statsmodels.api as sm
import scipy
from scipy import stats

# Process management
import shutil
import subprocess

# Workflow management
import papermill as pm

# Visualization
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

# Date and time
from datetime import datetime

# Get the timestamp
d = datetime.now().strftime("%m%d%Y")
t = datetime.now().strftime("%m%d%Y_%H%M%S")
print("Last iteration date:", d, "EST")
print("Last iteration time:", t, "EST")

# Get package versions
packages = [pd, np, matplotlib, sns, sm, scipy, pm]

print("\nPackage versions:\n")
for p in packages:
    print("-", p.__name__, p.__version__)

In [ ]:
%cd /home/jupyter/workspaces/itsn1burdenanalysis/121825_ITSN1_Results

## Papermill Configuration

In [ ]:
# parameters

# Gene name
GENE = "ITSN1"
gene = GENE         # alias

# Chromosome
CHROM = "chr21"
chrom = CHROM       # alias

# Start
STARTBP = str(33542501)
startbp = STARTBP   # alias

# End
ENDBP = str(33999861)
endbp = ENDBP       # alias

# Pick ANC for default
ANC = "EUR"

ancestry = ANC    
# Pick release

RELEASE = str(8)

release = RELEASE   # alias

# Data type

DT = "wgs"


In [ ]:
# Parameters
ANC = "EUR"
GENE = "ITSN1"
CHROM = "chr21"
STARTBP = "33542501"
ENDBP = "33999861"
RELEASE = "8"
DT = "wgs"
ancestry = "EUR"


In [ ]:
# Paths

# Home directory
HOME = "/home/jupyter/workspaces/itsn1burdenanalysis"

# Directory for base results
BASE_RESULTS_DIR = f"{HOME}/121825_ITSN1_Results"

# Data output directory (for WGS and imputed data)
#DT_DIR = f"{BASE_RESULTS_DIR}/{GENE}_AOU_R{RELEASE}"

# Work directory for current ANC
WORK_DIR = f"{BASE_RESULTS_DIR}/{ANC}_R{RELEASE}_{DT}"

# Default VEP directory
VEP_DIR='/home/jupyter/workspaces/itsn1burdenanalysis/ensembl-vep'

# Default workspace directory
DATA_DIR = f"{HOME}/121825_ITSN1_Results/data/"

#default tools dir
TOOLS_DIR='/home/jupyter/workspaces/itsn1burdenanalysis/tools'

# Create directories
dirsToCreate = [WORK_DIR]
for c in dirsToCreate:
    os.makedirs(c, exist_ok=True)
    

## Environment Configuration

In [ ]:
# create a results directory
bucket = os.getenv("WORKSPACE_BUCKET")

In [ ]:
# get fasta file references from hail

#! wget -O {DATA_DIR}/Homo_sapiens_assembly38.fasta.fai https://storage.googleapis.com/hail-common/references/Homo_sapiens_assembly38.fasta.fai
#! wget -O {DATA_DIR}/Homo_sapiens_assembly38.fasta.gz https://storage.googleapis.com/hail-common/references/Homo_sapiens_assembly38.fasta.gz

In [ ]:
#! gunzip {DATA_DIR}/Homo_sapiens_assembly38.fasta.gz

In [ ]:
! ls {BASE_RESULTS_DIR}

## Package Installs

### plink 1.9

In [ ]:
%%bash

mkdir -p /home/jupyter/workspaces/itsn1burdenanalysis/tools
cd /home/jupyter/workspaces/itsn1burdenanalysis/tools

if test -e /home/jupyter/workspaces/itsn1burdenanalysis/tools/plink; then
echo "Plink1.9 is already installed in ~/tools/"

else
echo -e "Downloading plink \n    -------"
wget -N http://s3.amazonaws.com/plink1-assets/plink_linux_x86_64_20190304.zip 
unzip -o plink_linux_x86_64_20190304.zip
echo -e "\n plink downloaded and unzipped in ~/tools \n "

fi

### plink 2

In [ ]:
%%bash

mkdir -p /home/jupyter/workspaces/itsn1burdenanalysis/tools
cd /home/jupyter/workspaces/itsn1burdenanalysis/tools

if test -e /home/jupyter/workspaces/itsn1burdenanalysis/tools/plink2; then
echo "Plink2 is already installed in /home/jupyter/tools/"

else
echo -e "Downloading plink2 \n    -------"
wget -N https://s3.amazonaws.com/plink2-assets/alpha5/plink2_linux_x86_64_20240820.zip
unzip -o plink2_linux_x86_64_20240820.zip
echo -e "\n plink2 downloaded and unzipped in /home/jupyter/tools \n "

fi

### rvtests

In [ ]:
%%bash

#Install RVTESTS: Option 1 (~15min)
if test -e /home/jupyter/workspaces/itsn1burdenanalysis/tools/rvtests; then

echo "rvtests is already installed"
else
echo "rvtests is not installed"

mkdir /home/jupyter/workspaces/itsn1burdenanalysis/tools/rvtests
cd //home/jupyter/workspaces/itsn1burdenanalysis/tools/rvtests

wget https://github.com/zhanxw/rvtests/releases/download/v2.1.0/rvtests_linux64.tar.gz 

tar -zxvf rvtests_linux64.tar.gz
fi

In [ ]:
# chmod to make sure you have permission to run the program
! chmod u+x /home/jupyter/workspaces/itsn1burdenanalysis/tools/plink
! chmod u+x /home/jupyter/workspaces/itsn1burdenanalysis/tools/plink2
! chmod 777 /home/jupyter/workspaces/itsn1burdenanalysis/tools/rvtests/executable/rvtest

## Read in Covariate File

In [ ]:
# read in covariate file
covar = pd.read_csv(f'{DATA_DIR}/covariate_wgs_R8.txt', sep='\t', header=0)
covar

In [ ]:
covar["Ancestry"].value_counts(dropna=False)

In [ ]:
# subset to ancestry for analysis

final_df = covar[covar["Ancestry"]==ancestry]
final_df

In [ ]:
samples_toKeep = final_df[['FID', 'IID']].copy()

samples_toKeep.to_csv(f'{WORK_DIR}/{ancestry}.samplestoKeep.txt', sep = '\t', index=False, header=None)
samples_toKeep.shape

In [ ]:
pheno = final_df[['FID', 'IID', 'PHENO']].copy()
pheno.to_csv(f'{WORK_DIR}/{ancestry}_pheno.txt', sep = '\t', index=False)

In [ ]:
# save covariate file
final_df.to_csv(f'{WORK_DIR}/{ancestry}_covariate_file.txt', sep = '\t', index=False, header=True, na_rep="NA")

In [ ]:
! ls {WORK_DIR}

### Get cohort summary stats

## Prepare PLINK input files

In [ ]:
## extract region of interest from file, flanked by 100kb on both sides
! {TOOLS_DIR}/plink2 \
--bfile {DATA_DIR}/chr21_{GENE}.mt \
--chr 21 \
--from-bp {STARTBP} \
--to-bp {ENDBP} \
--set-all-var-ids 'chr@:#:$r:$a' \
--new-id-max-allele-len 999 \
--fa {DATA_DIR}/Homo_sapiens_assembly38.fasta \
--make-pgen erase-dosage \
--out {WORK_DIR}/chr21_{GENE}

In [ ]:
# filter file down to ancestry and mac 1
! {TOOLS_DIR}/plink2 \
--pfile {WORK_DIR}/chr21_{GENE} \
--keep {WORK_DIR}/{ancestry}.samplestoKeep.txt \
--pheno {WORK_DIR}/{ancestry}_pheno.txt \
--mac 1 \
--make-pgen \
--out {WORK_DIR}/chr21_{ancestry}

In [ ]:
! ls {WORK_DIR}

## Create VCF file from PLINK pgen
- *ITSN1* from NCBI gene, with gene region flanked by 200kb on both sides
- hg38 (chr21:33442501-34099861) 

In [ ]:
# set file prefix
PLINK_PREFIX = f'chr21_{ancestry}'

In [ ]:
## generate individual pfile for variant annotation

# Get one sample 
samples = pd.read_csv(f"{WORK_DIR}/{ancestry}.samplestoKeep.txt", sep = "\t", header=None)
sample_id = samples.iloc[0,1]

# convert to vcf for annotation
! {TOOLS_DIR}/plink2 \
--pfile {WORK_DIR}/{PLINK_PREFIX} \
--recode vcf id-paste=iid \
--indv {sample_id} \
--silent \
--out {WORK_DIR}/{ancestry}_vep_input

In [ ]:
! ls {WORK_DIR}

In [ ]:
### Bgzip and Tabix (zip and index the file)
! bgzip -k -f {WORK_DIR}/{ancestry}_vep_input.vcf
! tabix -f -p vcf {WORK_DIR}/{ancestry}_vep_input.vcf.gz
! tabix -f -p vcf {WORK_DIR}/{ancestry}_vep_input.vcf.gz

In [ ]:
! ls

In [ ]:
### Bgzip and Tabix (zip and index the file)
! bgzip -f {WORK_DIR}/{PLINK_PREFIX}.vcf
! tabix -f -p vcf {WORK_DIR}/{PLINK_PREFIX}.vcf.gz

In [ ]:
# annotating just one sample id to make annotation run faster
sample_id = final_df["IID"].iloc[0]
filename = f'{WORK_DIR}/{PLINK_PREFIX}.vcf.gz'

## Annotation using VEP

In [ ]:
# run container to generate VEP output
vep_cmd = [
    f"{VEP_DIR}/vep",
    "--cache",
    "--offline",
    "--force_overwrite",
    "--input_file", f"{WORK_DIR}/{ancestry}_vep_input.vcf",
    "--output_file", f"{WORK_DIR}/{ancestry}_vep_output.txt",
    "--pick_allele",
    "--refseq",
    "--hgvs", # get hgvs annotations 
    "--use_transcript_ref",
    "--assembly", "GRCh38",
    "--fasta", "/home/jupyter/.vep/homo_sapiens/114_GRCh38/Homo_sapiens.GRCh38.dna.toplevel.fa.gz",
    "--force_overwrite"
]
result = subprocess.run(vep_cmd)
if result.returncode == 0:
    print("VEP annotation complete!")
else:
    print("VEP failed :(")

In [ ]:
# Remove header from vep output, subset to important columns and copy to work directory

! grep -v "##" {WORK_DIR}/{ancestry}_vep_output.txt | cut -f 1-7,14 > {WORK_DIR}/{ancestry}_vep_output_subset.txt

In [ ]:
# read in file from VEP filtered output
gene = pd.read_csv(f"{WORK_DIR}/{ancestry}_vep_output_subset.txt", sep="\t")

In [ ]:
gene.columns

In [ ]:
gene

In [ ]:
# ref seq ID for ITSN1 is 6453, filter for it to keep only ITSN1 variants
gene_subset =  gene[(gene["Gene"] == 6453) | (gene["Gene"] == "6453")].copy()
gene_subset

In [ ]:
# edit dataframe to create variants to keep file that matches plink format
gene_subset["Location"] = gene_subset["Location"].str.replace("21:", "")
variation_df = gene_subset["#Uploaded_variation"].str.split(":", expand=True)
location_df = gene_subset["Location"].str.split("-", expand=True)

location_df.columns = ["Start", "End"]
location_df["End"] = location_df.apply(lambda row: row["Start"] if row["End"] is None else row["End"], axis=1)

variation_df.columns = ["Chr", "Start", "Ref", "Alt"]
variation_df.drop(columns=["Start"], inplace=True)

fixed_columns = pd.concat([variation_df, location_df], axis=1)
gene_df = pd.concat([fixed_columns,gene_subset], axis=1)
hgvs_annotations = gene_df["Extra"].str.split(":", expand=True)
gene_df["Transcript Consequence"] = hgvs_annotations.iloc[:,1]
gene_df["Transcript Consequence"] = gene_df["Transcript Consequence"].str.split(';').str[0]


gene_df["Protein Consequence"] = hgvs_annotations.iloc[:,2]
# drop unnecessary columns
gene_df.drop(columns=["Location", "Feature", "Feature_type", "Extra"], axis=1, inplace=True)

gene_df["Gene"] = "ITSN1"
gene_df.rename(columns={"#Uploaded_variation":"SNP"}, inplace=True)
gene_df

In [ ]:
# get value counts to determine how many total annotated
gene_df["Consequence"].value_counts()

In [ ]:
# write value_counts output to file
gene_df["Consequence"].value_counts().to_csv(f"{WORK_DIR}/{ancestry}_{GENE}_all_vep_consequences.txt", sep="\t")

In [ ]:
# filter consequences to only missense, splicing, frameshift and stopgain, and other high priority consequences
# Source: https://useast.ensembl.org/info/genome/variation/prediction/predicted_data.html 

vep_coding_variants = gene_df[gene_df["Consequence"].str.contains('^splice_donor_variant|^splice_acceptor_variant|^missense|^stop_gained|frame|^protein_altering|transcript_ablation|transcript_amplification', regex=True)].copy()
vep_coding_variants["Consequence"].value_counts()

In [ ]:
# filter out any synonymous variants
final_vep_coding_variants = vep_coding_variants[~(vep_coding_variants["Consequence"].str.contains('synonymous_variant', regex=False))].copy()
final_vep_coding_variants["Consequence"].value_counts()

In [ ]:
# Save ids in PLINK format
final_vep_coding_variants["SNP"].to_csv(f"{WORK_DIR}/{ancestry}_{GENE}.all_variants_toKeep.txt", sep="\t",  index=False, header=False)

In [ ]:
# save consequences annotation file for later
final_vep_coding_variants.to_csv(f"{WORK_DIR}/{ancestry}_{GENE}_vep_coding_variant_annotations.txt",  sep="\t",  index=False)

In [ ]:
! ls {WORK_DIR}

## Burden Analyses using RVTests

In [ ]:
PFILE_PREFIX = f"{CHROM}_{ancestry}"

In [ ]:
# get hg38 refFlat file from ucsc
#! wget -P {DATA_DIR} https://hgdownload.soe.ucsc.edu/goldenPath/hg38/database/refFlat.txt.gz

In [ ]:
#! gunzip {DATA_DIR}/refFlat.txt.gz

In [ ]:
# Convert the files from Plink 2.0 to Plink 1.9 format 
! {TOOLS_DIR}/plink2 \
--pfile {WORK_DIR}/{PFILE_PREFIX} \
--make-bed \
--pheno {WORK_DIR}/{ancestry}_pheno.txt \
--pheno-name PHENO \
--max-alleles 2 \
--fa {DATA_DIR}/Homo_sapiens_assembly38.fasta \
--out {WORK_DIR}/chr21_{ancestry}

In [ ]:
## extract variants
! {TOOLS_DIR}/plink2 \
--pfile {WORK_DIR}/chr21_{ancestry} \
--keep {WORK_DIR}/{ancestry}.samplestoKeep.txt \
--extract {WORK_DIR}/{ancestry}_{GENE}.all_variants_toKeep.txt \
--recode vcf-iid \
--out {WORK_DIR}/{ancestry}_{GENE}.coding_nonsyn

In [ ]:
## Bgzip and Tabix (zip and index the file)
! bgzip -f {WORK_DIR}/{ancestry}_{GENE}.coding_nonsyn.vcf
! tabix -f -p vcf {WORK_DIR}/{ancestry}_{GENE}.coding_nonsyn.vcf.gz

In [ ]:
# because of errors with rvtests reading in phenotypes, create cov file that matches rvtest pheno file header
rv_df = pd.read_csv(f"{WORK_DIR}/{ancestry}_covariate_file.txt", sep="\t")
rv_df.columns = rv_df.columns.str.lower()
rv_df["fatid"] = 0
rv_df["matid"] = 0
rv_df = rv_df[["fid", "iid", "fatid", "matid", "sex", "pheno", "age", "pc1", "pc2", "pc3", "pc4", "pc5"]]
rv_df.to_csv(f'{WORK_DIR}/{ancestry}_rvtests_covariate_file.txt', sep='\t', index=False, header=True)

In [ ]:
## RVtests with covariates 
! {TOOLS_DIR}/rvtests/executable/rvtest --noweb --hide-covar \
--out {WORK_DIR}/{ancestry}_{GENE}.burden.coding_nonsyn \
--kernel skat,skato \
--inVcf {WORK_DIR}/{ancestry}_{GENE}.coding_nonsyn.vcf.gz \
--pheno {WORK_DIR}/{ancestry}_rvtests_covariate_file.txt \
--pheno-name pheno \
--gene {GENE} \
--geneFile {DATA_DIR}/refFlat.txt \
--covar {WORK_DIR}/{ancestry}_rvtests_covariate_file.txt \
--covar-name sex,pc1,pc2,pc3,pc4,pc5

# --out : Name of output 
# --burden cmc --kernel skato: tests to run 
# --inVcf : VCF file 
# --gene: gene name (if only looking at one or a few)
# --geneFile refFlat.txt
# --pheno :  covar file
# --mpheno : # column that has phenotype information
# --pheno-name : column name with phenotype in file
# --covar : covar file
# --freqUpper : optional, MAF cut-off
# --covar-name : covariates, listed by column name, separated by commas (no spaces between commas)
## 1=controls; 2=cases

In [ ]:
! cat {WORK_DIR}/{ancestry}_{GENE}.burden.coding_nonsyn.Skat.assoc

In [ ]:
! cat {WORK_DIR}/{ancestry}_{GENE}.burden.coding_nonsyn.SkatO.assoc

# Case/Control Frequencies

### GLM

In [ ]:
# run logistic regression for pvals and odds ratios
! {TOOLS_DIR}/plink2 \
--pfile {WORK_DIR}/{PLINK_PREFIX} \
--adjust \
--keep {WORK_DIR}/{ancestry}.samplestoKeep.txt \
--extract {WORK_DIR}/{ancestry}_{GENE}.all_variants_toKeep.txt \
--ci 0.95 \
--covar {WORK_DIR}/{ancestry}_covariate_file.txt \
--no-input-missing-phenotype \
--covar-name SEX,PC1,PC2,PC3,PC4,PC5 \
--covar-variance-standardize \
--glm hide-covar omit-ref firth-fallback cols=+a1freq,+a1freqcc,+a1count,+totallele,+a1countcc,+totallelecc,+gcountcc,+err \
--out {WORK_DIR}/{ancestry}_{GENE}

### Run basic association test, with fisher

In [ ]:
BFILE_PREFIX = f"{CHROM}_{ancestry}"

In [ ]:
## Run association test using Fisher's exact test
! {TOOLS_DIR}/plink \
--bfile {WORK_DIR}/{BFILE_PREFIX} \
--extract {WORK_DIR}/{ancestry}_{GENE}.all_variants_toKeep.txt \
--keep {WORK_DIR}/{ancestry}.samplestoKeep.txt \
--pheno {WORK_DIR}/{ancestry}_pheno.txt \
--mpheno 1 \
--assoc fisher \
--ci 0.95 \
--allow-no-sex \
--out {WORK_DIR}/{ancestry}_{GENE}

### Get variant frequencies

In [ ]:
! {TOOLS_DIR}/plink \
--bfile {WORK_DIR}/{BFILE_PREFIX} \
--pheno {WORK_DIR}/{ancestry}_pheno.txt \
--extract {WORK_DIR}/{ancestry}_{GENE}.all_variants_toKeep.txt \
--keep {WORK_DIR}/{ancestry}.samplestoKeep.txt \
--allow-no-sex \
--freq \
--out {WORK_DIR}/{ancestry}_{GENE}

In [ ]:
# get recode file for carrier information
! {TOOLS_DIR}/plink \
--bfile {WORK_DIR}/{BFILE_PREFIX} \
--extract {WORK_DIR}/{ancestry}_{GENE}.all_variants_toKeep.txt \
--keep {WORK_DIR}/{ancestry}.samplestoKeep.txt \
--pheno {WORK_DIR}/{ancestry}_pheno.txt \
--pheno-name PHENO \
--allow-no-sex \
--recode A \
--out {WORK_DIR}/{ancestry}_{GENE}

### merge output files together

In [ ]:
### merge output files

ASSOC_FILE = f"{WORK_DIR}/{ancestry}_{GENE}.assoc.fisher"
RECODE_FILE = f"{WORK_DIR}/{ancestry}_{GENE}.raw"
GLM_HYBRID_FILE = f"{WORK_DIR}/{ancestry}_{GENE}.PHENO.glm.logistic.hybrid"
GLM_ADJUST_FILE = f"{WORK_DIR}/{ancestry}_{GENE}.PHENO.glm.logistic.hybrid.adjusted"
FREQ_FILE = f"{WORK_DIR}/{ancestry}_{GENE}.frq"


log_hybrid = pd.read_csv(GLM_HYBRID_FILE, sep="\s+", usecols=["ID", "A1", "A1_FREQ", "A1_CT", "ALLELE_CT", "P", "OR", "LOG(OR)_SE", "L95", "U95", "FIRTH?", "ERRCODE"])
log_hybrid.rename(columns={"ID":"SNP"}, inplace=True)

assoc_adjusted = pd.read_csv(GLM_ADJUST_FILE,  sep="\s+", usecols=["ID", "BONF"])
assoc_adjusted.rename(columns={"ID":"SNP"}, inplace=True)

assoc = pd.read_csv(ASSOC_FILE, sep="\s+", usecols=["SNP", "P", "F_A", "F_U"])
assoc.rename(columns={"P": "P_fisher"}, inplace=True)


freq = pd.read_csv(FREQ_FILE, sep='\s+', usecols=['SNP', 'MAF'])
df = pd.merge(log_hybrid, assoc_adjusted, on="SNP", how="right")
df = pd.merge(df, freq, on="SNP", how="left")
#merge freq with df
freq_assoc = pd.merge(assoc, df, on="SNP", how="left")

#read in recode file
recode = pd.read_csv(RECODE_FILE, sep="\s+")

# Pre-filter the dataset
cases_data = recode[recode["PHENOTYPE"] == 2]
controls_data = recode[recode["PHENOTYPE"] == 1]
# Make a list from the column names
column_names = recode.columns.tolist()

# Drop the first 6 columns to keep the variants 
variants = column_names[6:]
results = []
for variant in variants:
    # For cases
    hom_cases = cases_data[cases_data[variant] == 2].shape[0]
    het_cases = cases_data[cases_data[variant] == 1].shape[0]
    total_cases = cases_data.shape[0]
    # For controls
    hom_controls = controls_data[controls_data[variant] == 2].shape[0]
    het_controls = controls_data[controls_data[variant] == 1].shape[0]
    total_controls = controls_data.shape[0]
    results.append({
        "Variant": variant,
        "Hom Cases": hom_cases,
        "Het Cases": het_cases,
        "Total Cases": total_cases,
        "Hom Controls": hom_controls,
        "Het Controls": het_controls,
        "Total Controls": total_controls,
    })

# Return results
df_results = pd.DataFrame(results)
df_results["SNP"] = df_results["Variant"].apply(lambda x: x.rsplit("_", 1)[0])
df_results = df_results.drop("Variant", axis=1)

# Merge with assoc results 
full_results = pd.merge(freq_assoc, df_results, on="SNP", how="left")


# append variant annotation
annotations = pd.read_csv(f"{WORK_DIR}/{ancestry}_{GENE}_vep_coding_variant_annotations.txt", sep="\t", usecols=["Consequence", "SNP", "Transcript Consequence", "Protein Consequence"])
full_results_annotated = pd.merge(full_results, annotations, on="SNP", how="left")
full_results_annotated.head()
#subset to only columns to keep

clean_full_results = full_results_annotated[["SNP", "Consequence", "Transcript Consequence", "Protein Consequence", "A1", "A1_CT", "A1_FREQ", "FIRTH?",
                                             "P", "P_fisher", "BONF", "OR", "LOG(OR)_SE","L95", "U95",
                                  "F_A", "F_U", "MAF",
                                   "Hom Cases", "Het Cases", "Total Cases", 
                                   "Hom Controls","Het Controls", "Total Controls", "ERRCODE"
                                   ]].copy()



clean_full_results.insert(1, "Ancestry", ancestry)

clean_full_results.head()

In [ ]:
# Look at significant SNPs, if any 
sig_freq = clean_full_results[clean_full_results['P']<0.05]
sig_snps = sig_freq['SNP'].tolist()
sig_df_results = clean_full_results[clean_full_results['SNP'].isin(sig_snps)]
sig_df_results

In [ ]:
# save files to working directory
clean_full_results.to_csv(f'{WORK_DIR}/{ancestry}_{GENE}_aou_R8.fullVariantInformation.txt', sep="\t", na_rep=".", index=False)
sig_df_results.to_csv(f'{WORK_DIR}/{ancestry}_{GENE}_aou_R8.SignificantVariantInformation.txt', sep="\t", na_rep=".", index=False)

## Case/Control Frequencies

###  Get adjusted pvalues for chi-squared assoc test p-vals

### Get variant frequencies

pd.set_option('display.max_rows', 20)
clean_full_results

## AAO Regression

! pwd

ancestry = 'afr'
GENE = 'ITSN1'
WORK_DIR =  f'/home/jupyter/workspaces/itsn1burdenanalysis/2025_{ancestry}_WGS_AoU_R8'
DATA_DIR = f'/home/jupyter/workspaces/itsn1burdenanalysis/data'
WORKSPACE_DIR = '/home/jupyter/workspaces/itsn1burdenanalysis'
TOOLS_DIR='/home/jupyter/workspaces/itsn1burdenanalysis/tools'

import pandas as pd
df = pd.read_csv(f'{WORK_DIR}/{ancestry}_{GENE}_AoU_R8.fullVariantInformation.txt', sep="\t")
pd.set_option('display.max_rows', None)

df